To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your own computer, follow the installation instructions on our Github page [here](https://docs.unsloth.ai/get-started/installing-+-updating).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### News

**Read our [Gemma 3 blog](https://unsloth.ai/blog/gemma3) for what's new in Unsloth and our [Reasoning blog](https://unsloth.ai/blog/r1-reasoning) on how to train reasoning models.**

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### Installation

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    !pip install --no-deps unsloth vllm

In [ ]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    !pip install --no-deps unsloth vllm
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    # Skip restarting message in Colab
    import sys, re, requests; modules = list(sys.modules.keys())
    for x in modules: sys.modules.pop(x) if "PIL" in x or "google" in x else None
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft "trl==0.15.2" triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer

    # vLLM requirements - vLLM breaks Colab due to reinstalling numpy
    f = requests.get("https://raw.githubusercontent.com/vllm-project/vllm/refs/heads/main/requirements/common.txt").content
    with open("vllm_requirements.txt", "wb") as file:
        file.write(re.sub(rb"(transformers|numpy|xformers)[^\n]{1,}\n", b"", f))
    !pip install -r vllm_requirements.txt

### Unsloth

Load up `Qwen 2.5 3B Instruct`, and set parameters

In [ ]:
!pip install vllm==0.8.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 109.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 MB 23.2 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: gguf
    Found existing installation: gguf 0.14.0
    Uninstalling gguf-0.14.0:
      Successfully uninstalled gguf-0.14.0
  Attempting uninstall: xformers
    Found existing installation: xformer

In [ ]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HUGGINGFACE_TOKEN')
login(hf_token)

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch
max_seq_length = 1024 # Can increase for longer reasoning traces
lora_rank = 16 # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Venassa/Qwen2.5-7B-Children_Narrative_Extraction-relation_extraction_GRPO",
    max_seq_length = max_seq_length,
    load_in_4bit = True, # False for LoRA 16bit
    fast_inference = True, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.7, # Reduce if out of memory
)


model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "gate_proj", "up_proj", "down_proj",
    ], # Remove QKVO if out of memory
    lora_alpha = lora_rank,
    use_gradient_checkpointing = "unsloth", # Enable long context finetuning
    random_state = 3407,
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 04-14 04:22:14 [__init__.py:239] Automatically detected platform cuda.
==((====))==  Unsloth 2025.3.19: Fast Qwen2 patching. Transformers: 4.50.3. vLLM: 0.8.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit with actual GPU utilization = 69.2%
Unsloth: Your GPU has CUDA compute capability 8.0 with VRAM = 39.56 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 1024. Num Sequences = 288.
Unsloth: vLLM's KV Cache can use up to

model-00002-of-00002.safetensors:   0%|          | 0.00/2.16G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

INFO 04-14 04:23:03 [weight_utils.py:281] Time spent downloading weights for unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit: 29.988430 seconds


model.safetensors.index.json:   0%|          | 0.00/112k [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/121M [00:00<?, ?B/s]

Not an error, but Unsloth cannot patch O projection layer with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2025.3.19 patched 28 layers with 28 QKV layers, 0 O layers and 28 MLP layers.
Unsloth: Already have LoRA adapters! We shall skip this step.


### Data Prep
<a name="Data"></a>

We directly leverage [@willccbb](https://gist.github.com/willccbb/4676755236bb08cab5f4e54a0475d6fb) for data prep and all reward functions. You are free to create your own!

In [ ]:
import re
from datasets import load_dataset, Dataset

# Load and prep dataset
SYSTEM_PROMPT = """
Respond in the following format:
<reasoning>
...
</reasoning>
<answer>
...
</answer>
"""

XML_COT_FORMAT = """\
<reasoning>
{reasoning}
</reasoning>
<answer>
{answer}
</answer>
"""

def extract_xml_answer(text: str) -> str:
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()

def extract_hash_answer(text: str) -> str | None:
    if "####" not in text:
        return None
    return text.split("####")[1].strip()

# uncomment middle messages for 1-shot prompting
def get_gsm8k_questions(split = "train") -> Dataset:
    data = load_dataset('openai/gsm8k', 'main')[split] # type: ignore
    data = data.map(lambda x: { # type: ignore
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': x['question']}
        ],
        'answer': extract_hash_answer(x['answer'])
    }) # type: ignore
    return data # type: ignore

dataset = get_gsm8k_questions()

# Reward functions
def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    q = prompts[0][-1]['content']
    extracted_responses = [extract_xml_answer(r) for r in responses]
    print('-'*20, f"Question:\n{q}", f"\nAnswer:\n{answer[0]}", f"\nResponse:\n{responses[0]}", f"\nExtracted:\n{extracted_responses[0]}")
    return [2.0 if r == a else 0.0 for r, a in zip(extracted_responses, answer)]

def int_reward_func(completions, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    extracted_responses = [extract_xml_answer(r) for r in responses]
    return [0.5 if r.isdigit() else 0.0 for r in extracted_responses]

def strict_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"^<reasoning>\n.*?\n</reasoning>\n<answer>\n.*?\n</answer>\n$"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

def soft_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"<reasoning>.*?</reasoning>\s*<answer>.*?</answer>"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

def count_xml(text) -> float:
    count = 0.0
    if text.count("<reasoning>\n") == 1:
        count += 0.125
    if text.count("\n</reasoning>\n") == 1:
        count += 0.125
    if text.count("\n<answer>\n") == 1:
        count += 0.125
        count -= len(text.split("\n</answer>\n")[-1])*0.001
    if text.count("\n</answer>") == 1:
        count += 0.125
        count -= (len(text.split("\n</answer>")[-1]) - 1)*0.001
    return count

def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    contents = [completion[0]["content"] for completion in completions]
    return [count_xml(c) for c in contents]

README.md:   0%|          | 0.00/7.94k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

# **自定义数据集 prompt设计+reward设计**

In [ ]:
from datasets import load_dataset,Dataset,concatenate_datasets

import json
import re
import random

In [ ]:
SYSTEM_PROMPT = """
你是一个儿童故事分析专家。请从儿童叙事文本判断事件1和事件2的关系类型：并列、动机-因果、心理-因果、物理-因果、使能-因果、无关系。事件五元组由(触发词；主语；宾语；时间状语；地点状语)组成，没有的部分为无

### 关系定义：
并列：事件1和事件2时间重叠
动机-因果：事件1是事件2的目标/计划
心理-因果：事件1的情绪/想法/心理引发事件2
物理-因果：事件1通过物理或自然规律导致事件2
使能-因果：无事件1则事件2不可能发生（需反事实验证）
无关系：无明显关联

### 判断步骤：
1.是否时间重叠？（→并列）
2.是否是目标/计划？（→动机-因果）
3.是否引发心理变化？（→心理-因果）
4.是否物理/自然规律导致？（→物理-因果）
5.若无前者，后者能发生吗？（不能→使能-因果）
6.若都不符合，则无关系。

严格按照以下格式输出，<推理过程></推理过程>为按步骤分析思考的内容，<答案></答案>为最终关系类型的结果。不要有多余的输出：
<推理过程>
按步骤分析判断依据（需体现6个判断步骤的逻辑）
</推理过程>
<答案>
关系类型
</答案>
"""

EXAMPLE_PROMPT = [
    {
        "role": "user",
        "content": "事件1：结果那个蜜蜂窝掉到地上。\n"
              "事件1五元组：(掉到；蜜蜂窝；地上；无；无)\n"
              "事件2：所有蜜蜂全都飞了出来。\n"
              "事件2五元组：(飞了出来；蜜蜂；无；无；无)\n"
    },
    {
        "role": "assistant",
        "content":
            "<推理过程>\n"
            "1.时间重叠检查：蜜蜂窝掉落与蜜蜂飞出是连续发生，但无明确时间重叠描述 → 排除并列\n"
            "2.动机分析：蜜蜂窝掉落不是蜜蜂飞出的目标或计划 → 排除动机-因果\n"
            "3.心理因素：无情绪或想法变化的描述 → 排除心理-因果\n"
            "4.物理规律验证：蜜蜂窝掉落导致巢穴破坏，蜜蜂被迫飞出符合物理因果关系\n"
            "5.反事实验证：若蜜蜂窝未掉落，蜜蜂仍可能留在巢内 → 使能因果不成立\n"
            "6.综合判断：符合物理因果定义\n"
            "</推理过程>\n"
            "<答案>\n"
            "物理-因果\n"
            "</答案>"
    },
    {
        "role": "user",
        "content": "事件1：夜晚小孩子和狗全睡着了。\n"
              "事件1五元组：(睡着；小孩子，狗；无；夜晚；无)\n"
              "事件2：那个青蛙趁夜色逃跑了。\n"
              "事件2五元组：(逃跑；青蛙；无；无；无)\n"
    },
    {
        "role": "assistant",
        "content":
            "<推理过程>\n"
            "1.时间重叠检查：两个事件均发生在夜晚，但未明确描述同时发生 → 排除并列\n"
            "2.动机分析：小孩子和狗睡着并非青蛙逃跑的目标 → 排除动机-因果\n"
            "3.心理因素：无心理活动或情绪变化描述 → 排除心理-因果\n"
            "4.物理规律验证：睡眠状态本身不构成物理因果关系 → 排除物理-因果\n"
            "5.反事实验证：若小孩和狗未入睡，可能阻止青蛙逃跑 → 睡眠状态使逃跑成为可能\n"
            "6.综合判断：事件1为事件2创造必要条件但非直接原因\n"
            "</推理过程>\n"
            "<答案>\n"
            "使能-因果\n"
            "</答案>"
    }
]


In [ ]:
def load_dataset(jsonl_path):
    data = []
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            # item["first_event"], item["second_event"], item["relation_type"]
            first_event = item["first_event"]
            second_event = item["second_event"]
            relation_type = item["relation_type"]

            sentence_text_1 = first_event["sentence_text"]
            event_tuple_1 = first_event["event_tuple"]
            sentence_text_2 = second_event["sentence_text"]
            event_tuple_2 = second_event["event_tuple"]

            text = (
            f"事件1：{sentence_text_1}\n"
            f"事件1五元组：{event_tuple_1}\n"
            f"事件2：{sentence_text_2}\n"
            f"事件2五元组：{event_tuple_2}\n"
            )

            data.append({
                "text": text,
                "relation_type": f"<推理过程>\n(这里是推理过程的内容)\n</推理过程>\n<答案>\n{relation_type}\n</答案>"
            })
    return data

In [ ]:
dataset = load_dataset("/content/output_relation_train.jsonl")
train_data = [{
    "prompt": [
        {"role": "system", "content": SYSTEM_PROMPT},
        *EXAMPLE_PROMPT,
        {"role": "user", "content": item["text"]}
    ],
    "answer": item["relation_type"]
} for item in dataset]

In [ ]:
print(train_data[0])

{'prompt': [{'role': 'system', 'content': '\n你是一个儿童故事分析专家。请从儿童叙事文本判断事件1和事件2的关系类型：并列、动机-因果、心理-因果、物理-因果、使能-因果、无关系。事件五元组由(触发词；主语；宾语；时间状语；地点状语)组成，没有的部分为无\n\n### 关系定义：\n并列：事件1和事件2时间重叠\n动机-因果：事件1是事件2的目标/计划\n心理-因果：事件1的情绪/想法/心理引发事件2\n物理-因果：事件1通过物理或自然规律导致事件2\n使能-因果：无事件1则事件2不可能发生（需反事实验证）\n无关系：无明显关联\n\n### 判断步骤：\n1.是否时间重叠？（→并列）\n2.是否是目标/计划？（→动机-因果）\n3.是否引发心理变化？（→心理-因果）\n4.是否物理/自然规律导致？（→物理-因果）\n5.若无前者，后者能发生吗？（不能→使能-因果）\n6.若都不符合，则无关系。\n\n严格按照以下格式输出，<推理过程></推理过程>为按步骤分析思考的内容，<答案></答案>为最终关系类型的结果。不要有多余的输出：\n<推理过程>\n按步骤分析判断依据（需体现6个判断步骤的逻辑）\n</推理过程>\n<答案>\n关系类型\n</答案>\n'}, {'role': 'user', 'content': '事件1：结果那个蜜蜂窝掉到地上。\n事件1五元组：(掉到；蜜蜂窝；地上；无；无)\n事件2：所有蜜蜂全都飞了出来。\n事件2五元组：(飞了出来；蜜蜂；无；无；无)\n'}, {'role': 'assistant', 'content': '<推理过程>\n1.时间重叠检查：蜜蜂窝掉落与蜜蜂飞出是连续发生，但无明确时间重叠描述 → 排除并列\n2.动机分析：蜜蜂窝掉落不是蜜蜂飞出的目标或计划 → 排除动机-因果\n3.心理因素：无情绪或想法变化的描述 → 排除心理-因果\n4.物理规律验证：蜜蜂窝掉落导致巢穴破坏，蜜蜂被迫飞出符合物理因果关系\n5.反事实验证：若蜜蜂窝未掉落，蜜蜂仍可能留在巢内 → 使能因果不成立\n6.综合判断：符合物理因果定义\n</推理过程>\n<答案>\n物理-因果\n</答案>'}, {'role': 'user', 'content': '事件1：夜晚小孩子和狗全睡着了。\n事件

In [ ]:
dataset_negative = load_dataset("/content/output_relation_train_negative.jsonl")
train_data_negative = [{
    "prompt": [
        {"role": "system", "content": SYSTEM_PROMPT},
        *EXAMPLE_PROMPT,
        {"role": "user", "content": item["text"]}
    ],
    "answer": item["relation_type"]
} for item in dataset_negative]

In [ ]:
random.shuffle(train_data_negative)

In [ ]:
train_data_negative = train_data_negative[:2000]

In [ ]:
train_data_negative[0]

{'prompt': [{'role': 'system',
   'content': '\n你是一个儿童故事分析专家。请从儿童叙事文本判断事件1和事件2的关系类型：并列、动机-因果、心理-因果、物理-因果、使能-因果、无关系。事件五元组由(触发词；主语；宾语；时间状语；地点状语)组成，没有的部分为无\n\n### 关系定义：\n并列：事件1和事件2时间重叠\n动机-因果：事件1是事件2的目标/计划\n心理-因果：事件1的情绪/想法/心理引发事件2\n物理-因果：事件1通过物理或自然规律导致事件2\n使能-因果：无事件1则事件2不可能发生（需反事实验证）\n无关系：无明显关联\n\n### 判断步骤：\n1.是否时间重叠？（→并列）\n2.是否是目标/计划？（→动机-因果）\n3.是否引发心理变化？（→心理-因果）\n4.是否物理/自然规律导致？（→物理-因果）\n5.若无前者，后者能发生吗？（不能→使能-因果）\n6.若都不符合，则无关系。\n\n严格按照以下格式输出，<推理过程></推理过程>为按步骤分析思考的内容，<答案></答案>为最终关系类型的结果。不要有多余的输出：\n<推理过程>\n按步骤分析判断依据（需体现6个判断步骤的逻辑）\n</推理过程>\n<答案>\n关系类型\n</答案>\n'},
  {'role': 'user',
   'content': '事件1：结果那个蜜蜂窝掉到地上。\n事件1五元组：(掉到；蜜蜂窝；地上；无；无)\n事件2：所有蜜蜂全都飞了出来。\n事件2五元组：(飞了出来；蜜蜂；无；无；无)\n'},
  {'role': 'assistant',
   'content': '<推理过程>\n1.时间重叠检查：蜜蜂窝掉落与蜜蜂飞出是连续发生，但无明确时间重叠描述 → 排除并列\n2.动机分析：蜜蜂窝掉落不是蜜蜂飞出的目标或计划 → 排除动机-因果\n3.心理因素：无情绪或想法变化的描述 → 排除心理-因果\n4.物理规律验证：蜜蜂窝掉落导致巢穴破坏，蜜蜂被迫飞出符合物理因果关系\n5.反事实验证：若蜜蜂窝未掉落，蜜蜂仍可能留在巢内 → 使能因果不成立\n6.综合判断：符合物理因果定义\n</推理过程>\n<答案>\n物理-因果\n</答案>'},
  {'role': 'user',
   'content': '事件

In [ ]:
dataset_train=Dataset.from_list(train_data)
dataset_train_negative=Dataset.from_list(train_data_negative)

In [ ]:
train = concatenate_datasets([dataset_train, dataset_train_negative])

In [ ]:
train_mini = train.select(range(500))

In [ ]:
len(train)

12502

# **奖励函数设计**

In [ ]:
!pip install fuzzywuzzy

In [ ]:
import re
from fuzzywuzzy import fuzz
import numpy as np

/usr/local/lib/python3.11/dist-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [ ]:
def calculate_reward(prompts, completions, answer, **kwargs) -> list[float]:
    """
    计算模型生成回答的奖励分数

    参数:
    - prompts: 输入的提示
    - completions: 模型生成的回答列表
    - answer: 训练数据的标注，包含推理过程和正确答案
    - kwargs: 额外参数

    返回:
    - 奖励分数列表，与completions列表一一对应
    """
    rewards = []

    # 处理每个模型生成的回答
    for completion, gold_answer in zip(completions, answer):
        ground_truth_relation = extract_relation_type(gold_answer)
        # print(f"标注结果",ground_truth_relation)

        content = completion[0]["content"].strip()
        # print(f"模型响应",content)
        # 初始化三部分奖励
        format_reward = 0.0
        reasoning_reward = 0.0
        accuracy_reward = 0.0
        # completion=completion[0]
        # 第一部分：检查输出格式（占40%权重）
        format_reward = check_output_format(content)
        # legacy_reward = legacy_tag_penalty(completion)

        # 第二部分：评估推理质量（占30%权重）
        reasoning_reward = evaluate_reasoning_quality(content)

        # 第三部分：检查答案正确性（占30%权重）
        accuracy_reward = check_answer_accuracy(content, ground_truth_relation)

        # 计算总奖励分数
        total_reward = 0.2 * format_reward + 0.3 * reasoning_reward + 0.5 * accuracy_reward

        # 将分数添加到结果列表
        rewards.append(total_reward)

    return rewards

In [ ]:
def extract_relation_type(answer_text):
    """从标准答案文本中提取关系类型"""
    try:
        # 提取<答案>标签中的内容
        answer_section = re.search(r'<答案>\s*(.*?)\s*</答案>', answer_text, re.DOTALL)
        if answer_section:
            relation_type = answer_section.group(1).strip()
            return relation_type
        return ""
    except:
        return ""

def check_output_format(completion) -> float:
    """检查输出是否符合要求的格式"""
    reward = 0.0

    required_tags = [
        ("<推理过程>", "</推理过程>", 0.5),
        ("<答案>", "</答案>", 0.5)
    ]

    for start_tag, end_tag, weight in required_tags:
        start_count = completion.count(start_tag)
        end_count = completion.count(end_tag)

        # 完全匹配且闭合
        if start_count == 1 and end_count == 1:
            content = completion.split(start_tag)[1].split(end_tag)[0].strip()
            if content:
                reward += weight
        # 存在但未闭合
        elif start_count > 0 or end_count > 0:
            reward += weight * 0.3  # 部分分数
        # 完全缺失
        else:
            pass

    return reward

def legacy_tag_penalty(completion) -> float:
    """检测并惩罚旧标签使用"""
    penalty = 0.0
    legacy_tags = ["<think>", "</think>"]

    for tag in legacy_tags:
        if tag in completion:
            penalty += 0.5  # 每个旧标签扣0.2分（总分最多扣0.4分）

    return -penalty  # 直接返回负值

In [ ]:
def evaluate_reasoning_quality(completion):
    """评估推理过程的质量，不要求严格的数字列表格式"""
    reward = 0.0

    # 提取推理过程
    reasoning_match = re.search(r'<推理过程>\s*(.*?)\s*</推理过程>', completion, re.DOTALL)
    if not reasoning_match:
        return 0.0

    reasoning = reasoning_match.group(1).strip()
    if not reasoning:
        return 0.0

    # 检查是否包含必要的思考步骤（无需按特定顺序或格式）
    reasoning_aspects = [
        # 并列关系的思考
        r'(时间重叠|并列|同时发生|时间上重合|同一时间)',

        # 动机-因果关系的思考
        r'(目标|计划|动机|意图|为了|目的|想要)',

        # 心理-因果关系的思考
        r'(心理|情绪|想法|感受|认为|感到|感觉)',

        # 物理-因果关系的思考
        r'(物理|自然规律|导致|引起|造成|产生|物理原因)',

        # 使能-因果关系的思考
        r'(条件|必要|无则不能|若无.*则|前提|使能|反事实)',

        # 综合判断或结论
        r'(综合|判断|结论|分析|最终|无关)'
    ]

    # 计算包含的思考方面数量
    aspect_count = 0
    aspect_matches = []
    for aspect_pattern in reasoning_aspects:
        match = re.search(aspect_pattern, reasoning, re.IGNORECASE)
        if match:
            aspect_count += 1
            aspect_matches.append(match.group(0))

    # 思考完整性分数
    reasoning_completeness = aspect_count / len(reasoning_aspects)

    # 检查推理过程中的逻辑连贯性
    # 分析段落结构和逻辑流
    sentences = re.split(r'[。！？\n]+', reasoning)
    sentences = [s.strip() for s in sentences if s.strip()]

    # 逻辑连接词检查
    logic_keywords = [
        '因为', '所以', '如果', '那么', '由于', '导致', '引起', '进而', '从而',
        '但是', '然而', '而且', '并且', '首先', '其次', '接着', '最后', '综上',
        '因此', '考虑到', '基于', '分析'
    ]

    # 计算使用逻辑连接词的句子比例
    sentences_with_logic = 0
    for sentence in sentences:
        if any(keyword in sentence for keyword in logic_keywords):
            sentences_with_logic += 1

    logic_coherence = 0.0
    if sentences:
        # 使用逻辑连接词的句子比例，至少要有40%的句子包含逻辑连接词
        logic_coherence = min(1.0, sentences_with_logic / max(1, len(sentences)) / 0.4)

    # 检查是否有明确的关系类型判断过程
    relation_judgement_keywords = {
        '并列': ['判断为并列', '属于并列', '是并列关系', '时间重叠'],
        '动机-因果': ['判断为动机', '属于动机', '是动机因果', '目标驱动'],
        '心理-因果': ['判断为心理', '属于心理', '是心理因果', '心理引发'],
        '物理-因果': ['判断为物理', '属于物理', '是物理因果', '物理导致'],
        '使能-因果': ['判断为使能', '属于使能', '是使能因果', '条件关系'],
        '无关系': ['判断为无关', '属于无关', '没有关系', '不相关']
    }

    # 检查是否有关系判断的明确表述
    identified_relations = []
    for relation_type, phrases in relation_judgement_keywords.items():
        if any(phrase in reasoning for phrase in phrases):
            identified_relations.append(relation_type)

    # 新增: 检查是否正确地排除了其他关系后才判断为"无关系"
    if '无关系' in identified_relations:
        # 检查是否明确排除了前五种关系
        has_exclusion_evidence = all(
            re.search(f'(不是|排除|不属于|不能判断为){relation_type}|{relation_type}(关系)?(不成立|不适用)',
                      reasoning, re.IGNORECASE)
            for relation_type in ['并列', '动机-因果', '心理-因果', '物理-因果', '使能-因果']
        )

        # 如果没有排除证据但判断为无关，则降低奖励
        if not has_exclusion_evidence:
            reasoning_completeness *= 0.5  # 降低思考完整性分数

    has_relation_judgement = len(identified_relations) > 0

    # 综合评分
    # 60% 权重给思考完整性
    # 30% 权重给逻辑连贯性
    # 10% 权重给是否有明确的关系判断
    reasoning_reward = (0.6 * reasoning_completeness +
                        0.3 * logic_coherence +
                        0.1 * (1.0 if has_relation_judgement else 0.0))

    return reasoning_reward

def check_answer_accuracy(completion, ground_truth):
    """检查答案是否正确"""
    # 提取模型生成的答案
    answer_match = re.search(r'<答案>\s*(.*?)\s*</答案>', completion, re.DOTALL)
    if not answer_match:
        return 0.0

    predicted_relation = answer_match.group(1).strip()

    # 对关系类型进行标准化处理
    predicted_relation = normalize_relation_type(predicted_relation)
    ground_truth = normalize_relation_type(ground_truth)

    # 检查答案是否完全正确
    if predicted_relation == ground_truth:
        return 1.0
    if ground_truth != "无关系" and predicted_relation == "无关系":
        return -0.5  # 错误预测负奖励

    # 不完全正确的情况下，计算相似度得分
    similarity = calculate_relation_similarity(predicted_relation, ground_truth)
    return 0.3 * similarity  # 最多给30%的部分分

def normalize_relation_type(relation):
    """标准化关系类型表述"""
    # 去除空格、换行等
    relation = re.sub(r'\s+', '', relation)

    # 处理常见的表达变体
    mapping = {
        '并列关系': '并列',
        '动机因果': '动机-因果',
        '动机因果关系': '动机-因果',
        '心理因果': '心理-因果',
        '心理因果关系': '心理-因果',
        '物理因果': '物理-因果',
        '物理因果关系': '物理-因果',
        '使能因果': '使能-因果',
        '使能因果关系': '使能-因果',
        '无明显关系': '无关系',
        '没有关系': '无关系'
    }

    for variant, standard in mapping.items():
        if variant in relation:
            return standard

    return relation

def calculate_relation_similarity(pred, truth):
    """计算两个关系类型之间的相似度"""
    # 关系类型相似度矩阵
    similarity_matrix = {
        '并列': {'并列': 1.0, '动机-因果': 0.1, '心理-因果': 0.1, '物理-因果': 0.1, '使能-因果': 0.1, '无关系': 0.0},
        '动机-因果': {'并列': 0.1, '动机-因果': 1.0, '心理-因果': 0.4, '物理-因果': 0.3, '使能-因果': 0.3, '无关系': 0.0},
        '心理-因果': {'并列': 0.1, '动机-因果': 0.4, '心理-因果': 1.0, '物理-因果': 0.3, '使能-因果': 0.3, '无关系': 0.0},
        '物理-因果': {'并列': 0.1, '动机-因果': 0.3, '心理-因果': 0.3, '物理-因果': 1.0, '使能-因果': 0.3, '无关系': 0.0},
        '使能-因果': {'并列': 0.1, '动机-因果': 0.3, '心理-因果': 0.3, '物理-因果': 0.3, '使能-因果': 1.0, '无关系': 0.0},
        '无关系': {'并列': 0.0, '动机-因果': 0.0, '心理-因果': 0.0, '物理-因果': 0.0, '使能-因果': 0.0, '无关系': 1.0}
    }

    # 如果某个关系类型不在矩阵中，返回0相似度
    return similarity_matrix.get(pred, {}).get(truth, 0.0)

<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations!

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import wandb
from google.colab import userdata

wnb_token=userdata.get('wandb_token')
wandb.login(key=wnb_token) # import wandb
run = wandb.init(
    project='GRPO-stage2-7B-test0414',
    entity='FeSCN',
    job_type="training",
    settings=wandb.Settings(init_timeout=120),
    anonymous="allow"
)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: yuxuan0612 (FeSCN) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    use_vllm = True, # use vLLM for fast inference!
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 5,
    bf16 = is_bfloat16_supported(),
    fp16 = not is_bfloat16_supported(),
    per_device_train_batch_size = 16,
    gradient_accumulation_steps = 2, # Increase to 4 for smoother training
    num_generations = 2, # Decrease if out of memory
    max_prompt_length = 512,
    max_completion_length = 384,
    num_train_epochs = 1, # Set to 1 for a full training run
    # max_steps = 100,
    save_steps = 200,
    max_grad_norm = 0.3,
    report_to = "wandb", # Can use Weights & Biases
    output_dir = "/content/drive/MyDrive/GRPO-stage2-7B-retrain-outputs",
)

And let's run the trainer! If you scroll up, you'll see a table of rewards. The goal is to see the `reward` column increase!

You might have to wait 150 to 200 steps for any action. You'll probably get 0 reward for the first 100 steps. Please be patient!

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |


In [ ]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        calculate_reward
    ],
    args = training_args,
    train_dataset = train,
)

In [ ]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 12,502 | Num Epochs = 1 | Total steps = 781
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 2 x 1) = 32
 "-____-"     Trainable parameters = 30,277,632/7,000,000,000 (0.43% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completion_length,kl,rewards / calculate_reward
5,0.003000,0.639667,0.156564,169.856250,0.075949,0.639667
10,0.003900,0.620828,0.147542,170.368750,0.097829,0.620828
15,0.003300,0.600386,0.159923,172.231250,0.083007,0.600386
20,0.002800,0.701605,0.134335,170.162500,0.070392,0.701605
25,0.003200,0.603344,0.138946,169.181250,0.078982,0.603344
30,0.003100,0.658766,0.145200,168.043750,0.077594,0.658766
35,0.003700,0.625266,0.119258,170.193750,0.092310,0.625266
40,0.003300,0.606900,0.120660,171.243750,0.081709,0.606900
45,0.003500,0.615344,0.147078,171.768750,0.087499,0.615344
50,0.003600,0.578321,0.157343,170.387500,0.090224,0.578321


TrainOutput(global_step=781, training_loss=0.003927483384362714, metrics={'train_runtime': 18808.4288, 'train_samples_per_second': 0.665, 'train_steps_per_second': 0.042, 'total_flos': 0.0, 'train_loss': 0.003927483384362714})

In [ ]:
wandb.finish()

train/completion_length,▃▅▅▇▅▃▄▅▄▇▄▁▄█▅▆▅▇▄▅▇▃▇▅▄▃▅▂▅▃▃▄▃▃▅▄▅▄▄▃
train/epoch,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▅▆▆▆▇▇▇▇▇▇██
train/global_step,▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▇▇▇▇▇▇▇█
train/grad_norm,▂▂▂▂▁▂▂▂▃▂▂▁▂▂▂▄▂▂▃█▂▂▁▂▁▃▂▃▃▁▃▂▃▂▂▂▂▃▃▃
train/kl,▂▁▂▃▄▅▅▆▁█▄▃▆▃▄▇▄▅▄▇▄▅▇▇▄▆▃▅▅▆▄▄▄▇▅▃▅▄▅▃
train/learning_rate,▃▅███████▇▇▆▆▆▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁
train/loss,▂▂▂▃▁▂▂▃▃▃▄▃▃▃▃▃▅▃█▃▄▃▃▃▄▃▃▅▄▃▅▂▃▂▃▃▃▃▃▃
train/reward,▂█▅▂▇▄▁▄▅▃▅▃▄▇▃▅▆▆▅▃▆▅▆▄▄▆▃▅▃▃▄▁▄▂▆▂▆▃▂▆
train/reward_std,▃▅▆▄█▇▂▅▅▆▆▇▆▄▆▅▅█▂▄▅▆▆▄▂▆▇▇▅▅▁▇▄▆▆▅▆▄▇▇
train/rewards/calculate_reward,▄▂▁▃▂▅▂▄▄▆▇▄▅▃▅▄▅▃▅▄▅▅▄▄▅▅▆▆▆▅▄▄▄█▄█▃▆▆▇
total_flos,0


<a name="Inference"></a>
### Inference
Now let's try the model we just trained! First, let's first try the model without any GRPO trained:

In [ ]:
text = tokenizer.apply_chat_template([
    {"role" : "user", "content" : "事件1：夜晚小孩子和狗全睡着了。"
                      "事件1五元组：(睡着; 小孩子，狗; 无; 夜晚; 无)"
                      "事件2：那个青蛙趁夜色逃跑了。"
                      "事件2五元组：(逃跑; 青蛙; 无; 无; 无)"},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 128,
)
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

output

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.88s/it, est. speed input: 53.65 toks/s, output: 67.99 toks/s]


'您给出的两个事件的五元组表示非常清晰，我们可以进一步分析这两个事件之间的关系或联系。为了更好地理解和分析这两个事件，我们可以从以下几个方面来探讨：\n\n1. **时间关联**：事件1描述的是夜晚，而事件2中的“夜色”也暗示了时间是在夜晚。这表明两个事件可能发生在同一时间段，但并没有直接的因果关系。\n\n2. **主体活动**：事件1中的主体是“小孩子和狗”，而事件2中的主体是“青蛙”。这两个事件描述了不同主体的活动，但也没有直接的因果关系。\n\n3. **环境因素'

And now with the LoRA we just trained with GRPO - we first save the LoRA first!

In [ ]:
model.save_lora("grpo_saved_lora")

Now we load the LoRA and test:

In [ ]:
text = tokenizer.apply_chat_template([
    {"role" : "system", "content" : SYSTEM_PROMPT},
    {"role" : "user", "content" : "事件1：夜晚小孩子和狗全睡着了。"
                      "事件1五元组：(睡着; 小孩子，狗; 无; 夜晚; 无)"
                      "事件2：那个青蛙趁夜色逃跑了。"
                      "事件2五元组：(逃跑; 青蛙; 无; 无; 无)"},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 256,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text

output

Processed prompts: 100%|██████████| 1/1 [00:04<00:00,  4.06s/it, est. speed input: 98.54 toks/s, output: 62.33 toks/s]


'<推理过程>\n1. 时间重叠：事件1描述的是夜晚，而事件2没有明确指出时间，但通常夜色逃走也发生在夜晚，故可能存在时间重叠，但并不确定是同一时间段，不足以判断为并列。\n2. 目标/计划：事件2中的青蛙逃跑并不是由事件1中的小孩子和狗睡着了所决定的，也不是目标或计划，故不符合动机-因果。\n3. 心理变化：事件1中的小孩子和狗睡着了并没有直接引发青蛙逃跑的心理变化，故不符合心理-因果。\n4. 物理/自然规律：青蛙的逃跑并不是因为小孩子和狗睡着了，而更可能是因为夜色提供了掩护，这属于物理条件，但直接因果关系不明显。\n5. 使能条件：如果小孩子和狗没有睡着，青蛙可能不会轻易逃跑。因此，小孩子的和狗的睡着为青蛙逃跑提供了条件，符合使能-因果。\n6. 无关系：根据以上分析，没有其他明显关系类型适用。\n\n综合以上分析，最符合的应该是使能-因果。\n</推理过程>\n<答案>\n使能-因果\n</答案>'

**保存推理结果**

In [ ]:
def run_inference(input_file, output_file, model, tokenizer):
    # 读取eval.jsonl文件
    with open(input_file, 'r', encoding='utf-8') as infile, open(output_file, 'w', encoding='utf-8') as outfile:
        for line_num, line in enumerate(infile, start=1):
            # 解析每一行的JSON数据
            data = json.loads(line)
            first_event = data["first_event"]
            second_event = data["second_event"]
            relation_type = data["relation_type"]

            sentence_text_1 = first_event["sentence_text"]
            event_tuple_1 = first_event["event_tuple"]
            sentence_text_2 = second_event["sentence_text"]
            event_tuple_2 = second_event["event_tuple"]

            text = (
            f"事件1：{sentence_text_1}\n"
            f"事件1五元组：{event_tuple_1}\n"
            f"事件2：{sentence_text_2}\n"
            f"事件2五元组：{event_tuple_2}\n"
            )

            # text = data.get('text', '')


            # 生成text
            text_input = tokenizer.apply_chat_template([
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": text},
            ], tokenize=False, add_generation_prompt=True)

            # 设置采样参数
            sampling_params = SamplingParams(
                temperature=0.8,
                top_p=0.95,
                max_tokens=512,
            )

            # 推理并获得输出
            output = model.fast_generate(
                text_input,
                sampling_params=sampling_params,
                lora_request=model.load_lora("grpo_saved_lora"),

            )[0].outputs[0].text

            # 创建新的输出数据
            result = {
                'id': line_num,
                'output': output
            }

            # 将结果写入到新的jsonl文件中
            outfile.write(json.dumps(result, ensure_ascii=False) + '\n')


In [ ]:
run_inference(input_file="/content/output_relation_eval.jsonl", output_file="/content/drive/MyDrive/two stage results/stage two/GRPO_1epoch_result_retrain.jsonl", model=model, tokenizer=tokenizer)

[large output cleared]


Our reasoning model is much better - it's not always correct, since we only trained it for an hour or so - it'll be better if we extend the sequence length and train for longer!

<a name="Save"></a>
### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False: model.save_pretrained_merged("model", tokenizer, save_method = "lora",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "lora", token = "")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "hf/model", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "",
    )

Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in llama.cpp or a UI based system like Jan or Open WebUI. You can install Jan [here](https://github.com/janhq/jan) and Open WebUI [here](https://github.com/open-webui/open-webui)

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>


# **保存到huggingface**

In [ ]:
from huggingface_hub import HfApi

# 首先手动创建一个 repo（在Huggingface网页上新建 Model Repo），或者代码创建：
from huggingface_hub import create_repo

repo_name = "Venassa/Qwen2.5-7B-Children_Narrative_Extraction-relation_extraction_GRPO_retrain"
create_repo(repo_name, exist_ok=True)

# push 上传
model.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)

adapter_model.safetensors:   0%|          | 0.00/121M [00:00<?, ?B/s]

Saved model to https://huggingface.co/Venassa/Qwen2.5-7B-Children_Narrative_Extraction-relation_extraction_GRPO_retrain


README.md:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

# 调用模型进行推理

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch
max_seq_length = 1024 # Can increase for longer reasoning traces
lora_rank = 16 # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Venassa/Qwen2.5-7B-Children_Narrative_Extraction-relation_extraction_GRPO",
    max_seq_length = max_seq_length,
    load_in_4bit = True, # False for LoRA 16bit
    fast_inference = True, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.7, # Reduce if out of memory
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 04-14 02:26:39 [__init__.py:239] Automatically detected platform cuda.
==((====))==  Unsloth 2025.3.19: Fast Qwen2 patching. Transformers: 4.50.3. vLLM: 0.8.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit with actual GPU utilization = 69.2%
Unsloth: Your GPU has CUDA compute capability 8.0 with VRAM = 39.56 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 1024. Num Sequences = 288.
Unsloth: vLLM's KV Cache can use up to

model-00002-of-00002.safetensors:   0%|          | 0.00/2.16G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

INFO 04-14 02:27:22 [weight_utils.py:281] Time spent downloading weights for unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit: 23.063522 seconds


model.safetensors.index.json:   0%|          | 0.00/112k [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/121M [00:00<?, ?B/s]

Not an error, but Unsloth cannot patch O projection layer with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2025.3.19 patched 28 layers with 28 QKV layers, 0 O layers and 28 MLP layers.


In [ ]:
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(152064, 3584, padding_idx=151654)
        (layers): ModuleList(
          (0-1): 2 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
              (k_proj): Linear(in_features=3584, out_features=512, bias=True)
              (v_proj): Linear(in_features=3584, out_features=512, bias=True)
              (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
              (rotary_emb): LlamaRotaryEmbedding()
            )
            (mlp): Qwen2MLP(
              (gate_proj): lora.Linear(
                (base_layer): Linear4bit(in_features=3584, out_features=18944, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linea

In [ ]:
from vllm import SamplingParams

In [ ]:
run_inference(input_file="/content/output_relation_eval.jsonl", output_file="/content/drive/MyDrive/two stage results/stage two/GRPO_1epoch_result_longer.jsonl", model=model, tokenizer=tokenizer)

[large output cleared]
